# OpLog Anatomy

Audits per-pass layer records, including tensor, graph, module, timing, and gradient metadata.

Every `COVERAGE_CALLS` entry below is resolved against the current TorchLens API in this notebook.

In [ ]:
from __future__ import annotations

import shutil
import sys
import tempfile
from pathlib import Path

import torch

PROJECT_ROOT = next(
    candidate
    for candidate in [Path.cwd(), *Path.cwd().parents]
    if (candidate / "torchlens").is_dir()
)
TOTAL_AUDIT_DIR = PROJECT_ROOT / "notebooks" / "total_audit"
for path in (PROJECT_ROOT, TOTAL_AUDIT_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import torchlens as tl  # noqa: E402
from _shared import (  # noqa: E402
    inline_show,
    make_clean_corrupt_pair,
    pretty_print_fields,
    tiny_branched_model,
    tiny_cnn,
    tiny_dynamic_model,
    tiny_model,
    tiny_recurrent,
    tiny_transformer,
)

torch.manual_seed(0)
TMPDIR = Path(tempfile.mkdtemp(prefix="torchlens-total-audit-"))
print(f"torchlens {tl.__version__}; __all__={len(tl.__all__)}; tmp={TMPDIR.name}")

In [ ]:
from torch import nn
from _shared import audit_touch_items, summarize_value  # noqa: E402


class AuditBufferModel(nn.Module):
    """Tiny model with a registered buffer for BufferLog coverage."""

    def __init__(self) -> None:
        """Initialize one linear layer and one buffer."""

        super().__init__()
        self.linear = nn.Linear(4, 2)
        self.register_buffer("offset", torch.ones(1, 4))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Apply the buffer before the linear layer."""

        return self.linear(x + self.offset)


model = tiny_model(seed=0).train()
x = torch.randn(1, 4, requires_grad=True)
log = tl.trace(
    model,
    x,
    layers_to_save="all",
    save_gradients=True,
    save_function_args=True,
    save_rng_states=True,
    train_mode=True,
    vis_opt="none",
)
loss = log.layer_list[-1].activation.sum()
log.log_backward(loss)
layer_pass = log.layer_list[1]
layer_log = log.layer_logs[layer_pass.layer_label_no_pass]
module_log = next(iter(log.modules))
module_pass = next(iter(module_log.passes.values()))
param_log = next(iter(log.params))
grad_fn_log = next(iter(log.grad_fns))
grad_fn_pass = next(iter(grad_fn_log.passes.values()))

buffer_log_model = AuditBufferModel().eval()
buffer_log_capture = tl.trace(
    buffer_log_model, torch.randn(1, 4), layers_to_save="all", vis_opt="none"
)
buffer_log = next(iter(buffer_log_capture.buffers))

clean, corrupt = make_clean_corrupt_pair(seed=0)
bundle = tl.bundle(
    [
        ("baseline", log),
        ("corrupt", tl.trace(tiny_model(seed=1).eval(), corrupt, vis_opt="none")),
    ],
    baseline="baseline",
)
AUDIT_CONTEXT = {
    "tl": tl,
    "Trace": log,
    "LayerLog": layer_log,
    "OpLog": layer_pass,
    "ModuleLog": module_log,
    "ModuleCallLog": module_pass,
    "ParamLog": param_log,
    "BufferLog": buffer_log,
    "GradFnLog": grad_fn_log,
    "GradFnCallLog": grad_fn_pass,
    "Bundle": bundle,
}
print(
    f"audit context: {len(log.layer_list)} layer passes, {len(log.modules)} modules, {len(log.grad_fns)} grad fns"
)

## OpLog Anatomy: concrete workflow

This cell demonstrates the user-facing workflow for this notebook before the manifest sweep.

In [ ]:
COVERAGE_CALLS = [
    "tl.OpLog",
    "tl.OpLog.DEFAULT_FILL_STATE",
    "tl.OpLog.FORK_POLICY",
    "tl.OpLog.PORTABLE_STATE_SPEC",
    "tl.OpLog._construction_done",
    "tl.OpLog._pass_finished",
    "tl.OpLog.activation",
    "tl.OpLog.activation_postfunc",
    "tl.OpLog.activation_transform",
    "tl.OpLog.args_captured",
    "tl.OpLog.autograd_saved_bytes",
    "tl.OpLog.autograd_saved_tensor_count",
    "tl.OpLog.bool_conditional_id",
    "tl.OpLog.bool_context_kind",
    "tl.OpLog.bool_is_branch",
    "tl.OpLog.bool_wrapper_kind",
    "tl.OpLog.buffer_address",
    "tl.OpLog.buffer_parent",
    "tl.OpLog.buffer_pass",
    "tl.OpLog.bytes_delta_at_call",
    "tl.OpLog.bytes_peak_at_call",
    "tl.OpLog.captured_arg_template",
    "tl.OpLog.captured_args",
    "tl.OpLog.captured_kwarg_template",
    "tl.OpLog.captured_kwargs",
    "tl.OpLog.child_layers",
    "tl.OpLog.children_tensor_versions",
    "tl.OpLog.co_parent_layers",
    "tl.OpLog.cond_branch_children_by_cond",
    "tl.OpLog.cond_branch_elif_children",
    "tl.OpLog.cond_branch_else_children",
    "tl.OpLog.cond_branch_start_children",
    "tl.OpLog.cond_branch_then_children",
    "tl.OpLog.conditional_branch_depth",
    "tl.OpLog.conditional_branch_stack",
    "tl.OpLog.container_spec",
    "tl.OpLog.containing_module",
    "tl.OpLog.containing_modules",
    "tl.OpLog.copy",
    "tl.OpLog.corresponding_grad_fn",
    "tl.OpLog.creation_order",
    "tl.OpLog.detach_saved_tensor",
    "tl.OpLog.edge_uses",
    "tl.OpLog.equivalent_operations",
    "tl.OpLog.extra_data",
    "tl.OpLog.feeds_output",
    "tl.OpLog.flops_backward",
    "tl.OpLog.flops_forward",
    "tl.OpLog.func_applied",
    "tl.OpLog.func_argnames",
    "tl.OpLog.func_autocast_state",
    "tl.OpLog.func_call_id",
    "tl.OpLog.func_call_stack",
    "tl.OpLog.func_config",
    "tl.OpLog.func_is_inplace",
    "tl.OpLog.func_kwargs_non_tensor",
    "tl.OpLog.func_name",
    "tl.OpLog.func_non_tensor_args",
    "tl.OpLog.func_positional_args_non_tensor",
    "tl.OpLog.func_rng_states",
    "tl.OpLog.func_time",
    "tl.OpLog.get_child_layers",
    "tl.OpLog.get_parent_layers",
    "tl.OpLog.grad_dtype",
    "tl.OpLog.grad_fn_id",
    "tl.OpLog.grad_fn_name",
    "tl.OpLog.grad_fn_object",
    "tl.OpLog.grad_memory",
    "tl.OpLog.grad_memory_str",
    "tl.OpLog.grad_shape",
    "tl.OpLog.gradient",
    "tl.OpLog.has_child_tensor_variations",
    "tl.OpLog.has_children",
    "tl.OpLog.has_co_parents",
    "tl.OpLog.has_gradient",
    "tl.OpLog.has_input_ancestor",
    "tl.OpLog.has_internally_initialized_ancestor",
    "tl.OpLog.has_parents",
    "tl.OpLog.has_saved_activations",
    "tl.OpLog.has_siblings",
    "tl.OpLog.in_cond_branch",
    "tl.OpLog.input_ancestors",
    "tl.OpLog.internally_initialized_ancestors",
    "tl.OpLog.internally_initialized_parents",
    "tl.OpLog.intervention_log",
    "tl.OpLog.io_role",
    "tl.OpLog.is_buffer_layer",
    "tl.OpLog.is_computed_inside_submodule",
    "tl.OpLog.is_final_output",
    "tl.OpLog.is_input_layer",
    "tl.OpLog.is_internally_initialized",
    "tl.OpLog.is_internally_terminated",
    "tl.OpLog.is_leaf_module_output",
    "tl.OpLog.is_output_ancestor",
    "tl.OpLog.is_output_layer",
    "tl.OpLog.is_part_of_iterable_output",
    "tl.OpLog.is_scalar_bool",
    "tl.OpLog.is_submodule_input",
    "tl.OpLog.is_submodule_output",
    "tl.OpLog.is_terminal_bool_layer",
    "tl.OpLog.iterable_output_index",
    "tl.OpLog.layer_label",
    "tl.OpLog.layer_label_no_pass",
    "tl.OpLog.layer_label_no_pass_short",
    "tl.OpLog.layer_label_raw",
    "tl.OpLog.layer_label_short",
    "tl.OpLog.layer_label_w_pass",
    "tl.OpLog.layer_label_w_pass_short",
    "tl.OpLog.layer_total_num",
    "tl.OpLog.layer_type",
    "tl.OpLog.layer_type_num",
    "tl.OpLog.leaf_module_pass",
    "tl.OpLog.log_tensor_grad",
    "tl.OpLog.lookup_keys",
    "tl.OpLog.macs_backward",
    "tl.OpLog.macs_forward",
    "tl.OpLog.materialize_activation",
    "tl.OpLog.materialize_gradient",
    "tl.OpLog.max_distance_from_input",
    "tl.OpLog.max_distance_from_output",
    "tl.OpLog.min_distance_from_input",
    "tl.OpLog.min_distance_from_output",
    "tl.OpLog.module_address_normalized",
    "tl.OpLog.module_entry_exit_thread_output",
    "tl.OpLog.module_entry_exit_threads_inputs",
    "tl.OpLog.module_nesting_depth",
    "tl.OpLog.module_passes_entered",
    "tl.OpLog.module_passes_exited",
    "tl.OpLog.modules_entered",
    "tl.OpLog.modules_entered_argnames",
    "tl.OpLog.modules_exited",
    "tl.OpLog.num_args",
    "tl.OpLog.num_keyword_args",
    "tl.OpLog.num_param_tensors",
    "tl.OpLog.num_params_frozen",
    "tl.OpLog.num_params_total",
    "tl.OpLog.num_params_trainable",
    "tl.OpLog.num_passes",
    "tl.OpLog.num_positional_args",
    "tl.OpLog.operation_equivalence_type",
    "tl.OpLog.operation_num",
    "tl.OpLog.output_descendants",
    "tl.OpLog.output_device",
    "tl.OpLog.output_path",
    "tl.OpLog.params",
    "tl.OpLog.params_memory",
    "tl.OpLog.params_memory_str",
    "tl.OpLog.parent_layer_arg_locs",
    "tl.OpLog.parent_layers",
    "tl.OpLog.parent_param_barcodes",
    "tl.OpLog.parent_param_logs",
    "tl.OpLog.parent_param_passes",
    "tl.OpLog.parent_param_shapes",
    "tl.OpLog.parent_params",
    "tl.OpLog.pass_num",
    "tl.OpLog.passes",
    "tl.OpLog.recurrent_group",
    "tl.OpLog.root_ancestors",
    "tl.OpLog.save_gradients",
    "tl.OpLog.save_tensor_data",
    "tl.OpLog.scalar_bool_value",
    "tl.OpLog.show",
    "tl.OpLog.sibling_layers",
    "tl.OpLog.source_trace",
    "tl.OpLog.tensor",
    "tl.OpLog.tensor_dtype",
    "tl.OpLog.tensor_label_raw",
    "tl.OpLog.tensor_memory",
    "tl.OpLog.tensor_memory_str",
    "tl.OpLog.tensor_shape",
    "tl.OpLog.transformed_activation",
    "tl.OpLog.transformed_activation_dtype",
    "tl.OpLog.transformed_activation_memory",
    "tl.OpLog.transformed_activation_shape",
    "tl.OpLog.transformed_gradient",
    "tl.OpLog.transformed_gradient_dtype",
    "tl.OpLog.transformed_gradient_memory",
    "tl.OpLog.transformed_gradient_shape",
    "tl.OpLog.uses_params",
]

pretty_print_fields(
    layer_pass,
    [
        "layer_label",
        "layer_label_w_pass",
        "pass_num",
        "func_name",
        "tensor_shape",
        "tensor_dtype",
        "has_gradient",
        "flops_forward",
        "macs_forward",
    ],
)
print("copy type", type(layer_pass.copy()).__name__)

## Identity and tensor data

The helper resolves each listed name against live notebook objects and prints compact samples.

In [ ]:
ITEMS = [
    "tl.OpLog",
    "tl.OpLog.DEFAULT_FILL_STATE",
    "tl.OpLog.FORK_POLICY",
    "tl.OpLog.PORTABLE_STATE_SPEC",
    "tl.OpLog._construction_done",
    "tl.OpLog._pass_finished",
    "tl.OpLog.activation",
    "tl.OpLog.activation_postfunc",
    "tl.OpLog.activation_transform",
    "tl.OpLog.args_captured",
    "tl.OpLog.autograd_saved_bytes",
    "tl.OpLog.autograd_saved_tensor_count",
    "tl.OpLog.bool_conditional_id",
    "tl.OpLog.bool_context_kind",
    "tl.OpLog.bool_is_branch",
    "tl.OpLog.bool_wrapper_kind",
    "tl.OpLog.buffer_address",
    "tl.OpLog.buffer_parent",
    "tl.OpLog.buffer_pass",
    "tl.OpLog.bytes_delta_at_call",
    "tl.OpLog.bytes_peak_at_call",
    "tl.OpLog.captured_arg_template",
    "tl.OpLog.captured_args",
    "tl.OpLog.captured_kwarg_template",
    "tl.OpLog.captured_kwargs",
    "tl.OpLog.child_layers",
]

audit_touch_items("Identity and tensor data", ITEMS, AUDIT_CONTEXT)

## Function metadata

The helper resolves each listed name against live notebook objects and prints compact samples.

In [ ]:
ITEMS = [
    "tl.OpLog.children_tensor_versions",
    "tl.OpLog.co_parent_layers",
    "tl.OpLog.cond_branch_children_by_cond",
    "tl.OpLog.cond_branch_elif_children",
    "tl.OpLog.cond_branch_else_children",
    "tl.OpLog.cond_branch_start_children",
    "tl.OpLog.cond_branch_then_children",
    "tl.OpLog.conditional_branch_depth",
    "tl.OpLog.conditional_branch_stack",
    "tl.OpLog.container_spec",
    "tl.OpLog.containing_module",
    "tl.OpLog.containing_modules",
    "tl.OpLog.copy",
    "tl.OpLog.corresponding_grad_fn",
    "tl.OpLog.creation_order",
    "tl.OpLog.detach_saved_tensor",
    "tl.OpLog.edge_uses",
    "tl.OpLog.equivalent_operations",
    "tl.OpLog.extra_data",
    "tl.OpLog.feeds_output",
    "tl.OpLog.flops_backward",
    "tl.OpLog.flops_forward",
    "tl.OpLog.func_applied",
    "tl.OpLog.func_argnames",
    "tl.OpLog.func_autocast_state",
    "tl.OpLog.func_call_id",
]

audit_touch_items("Function metadata", ITEMS, AUDIT_CONTEXT)

## Autograd and gradients

The helper resolves each listed name against live notebook objects and prints compact samples.

In [ ]:
ITEMS = [
    "tl.OpLog.func_call_stack",
    "tl.OpLog.func_config",
    "tl.OpLog.func_is_inplace",
    "tl.OpLog.func_kwargs_non_tensor",
    "tl.OpLog.func_name",
    "tl.OpLog.func_non_tensor_args",
    "tl.OpLog.func_positional_args_non_tensor",
    "tl.OpLog.func_rng_states",
    "tl.OpLog.func_time",
    "tl.OpLog.get_child_layers",
    "tl.OpLog.get_parent_layers",
    "tl.OpLog.grad_dtype",
    "tl.OpLog.grad_fn_id",
    "tl.OpLog.grad_fn_name",
    "tl.OpLog.grad_fn_object",
    "tl.OpLog.grad_memory",
    "tl.OpLog.grad_memory_str",
    "tl.OpLog.grad_shape",
    "tl.OpLog.gradient",
    "tl.OpLog.has_child_tensor_variations",
    "tl.OpLog.has_children",
    "tl.OpLog.has_co_parents",
    "tl.OpLog.has_gradient",
    "tl.OpLog.has_input_ancestor",
    "tl.OpLog.has_internally_initialized_ancestor",
    "tl.OpLog.has_parents",
]

audit_touch_items("Autograd and gradients", ITEMS, AUDIT_CONTEXT)

## Graph relationships

The helper resolves each listed name against live notebook objects and prints compact samples.

In [ ]:
ITEMS = [
    "tl.OpLog.has_saved_activations",
    "tl.OpLog.has_siblings",
    "tl.OpLog.in_cond_branch",
    "tl.OpLog.input_ancestors",
    "tl.OpLog.internally_initialized_ancestors",
    "tl.OpLog.internally_initialized_parents",
    "tl.OpLog.intervention_log",
    "tl.OpLog.io_role",
    "tl.OpLog.is_buffer_layer",
    "tl.OpLog.is_computed_inside_submodule",
    "tl.OpLog.is_final_output",
    "tl.OpLog.is_input_layer",
    "tl.OpLog.is_internally_initialized",
    "tl.OpLog.is_internally_terminated",
    "tl.OpLog.is_leaf_module_output",
    "tl.OpLog.is_output_ancestor",
    "tl.OpLog.is_output_layer",
    "tl.OpLog.is_part_of_iterable_output",
    "tl.OpLog.is_scalar_bool",
    "tl.OpLog.is_submodule_input",
    "tl.OpLog.is_submodule_output",
    "tl.OpLog.is_terminal_bool_layer",
    "tl.OpLog.iterable_output_index",
    "tl.OpLog.layer_label",
    "tl.OpLog.layer_label_no_pass",
    "tl.OpLog.layer_label_no_pass_short",
]

audit_touch_items("Graph relationships", ITEMS, AUDIT_CONTEXT)

## Module metadata

The helper resolves each listed name against live notebook objects and prints compact samples.

In [ ]:
ITEMS = [
    "tl.OpLog.layer_label_raw",
    "tl.OpLog.layer_label_short",
    "tl.OpLog.layer_label_w_pass",
    "tl.OpLog.layer_label_w_pass_short",
    "tl.OpLog.layer_total_num",
    "tl.OpLog.layer_type",
    "tl.OpLog.layer_type_num",
    "tl.OpLog.leaf_module_pass",
    "tl.OpLog.log_tensor_grad",
    "tl.OpLog.lookup_keys",
    "tl.OpLog.macs_backward",
    "tl.OpLog.macs_forward",
    "tl.OpLog.materialize_activation",
    "tl.OpLog.materialize_gradient",
    "tl.OpLog.max_distance_from_input",
    "tl.OpLog.max_distance_from_output",
    "tl.OpLog.min_distance_from_input",
    "tl.OpLog.min_distance_from_output",
    "tl.OpLog.module_address_normalized",
    "tl.OpLog.module_entry_exit_thread_output",
    "tl.OpLog.module_entry_exit_threads_inputs",
    "tl.OpLog.module_nesting_depth",
    "tl.OpLog.module_passes_entered",
    "tl.OpLog.module_passes_exited",
    "tl.OpLog.modules_entered",
    "tl.OpLog.modules_entered_argnames",
]

audit_touch_items("Module metadata", ITEMS, AUDIT_CONTEXT)

## Control-flow and buffering

The helper resolves each listed name against live notebook objects and prints compact samples.

In [ ]:
ITEMS = [
    "tl.OpLog.modules_exited",
    "tl.OpLog.num_args",
    "tl.OpLog.num_keyword_args",
    "tl.OpLog.num_param_tensors",
    "tl.OpLog.num_params_frozen",
    "tl.OpLog.num_params_total",
    "tl.OpLog.num_params_trainable",
    "tl.OpLog.num_passes",
    "tl.OpLog.num_positional_args",
    "tl.OpLog.operation_equivalence_type",
    "tl.OpLog.operation_num",
    "tl.OpLog.output_descendants",
    "tl.OpLog.output_device",
    "tl.OpLog.output_path",
    "tl.OpLog.params",
    "tl.OpLog.params_memory",
    "tl.OpLog.params_memory_str",
    "tl.OpLog.parent_layer_arg_locs",
    "tl.OpLog.parent_layers",
    "tl.OpLog.parent_param_barcodes",
    "tl.OpLog.parent_param_logs",
    "tl.OpLog.parent_param_passes",
    "tl.OpLog.parent_param_shapes",
    "tl.OpLog.parent_params",
    "tl.OpLog.pass_num",
    "tl.OpLog.passes",
]

audit_touch_items("Control-flow and buffering", ITEMS, AUDIT_CONTEXT)

## Remaining pass fields

The helper resolves each listed name against live notebook objects and prints compact samples.

In [ ]:
ITEMS = [
    "tl.OpLog.recurrent_group",
    "tl.OpLog.root_ancestors",
    "tl.OpLog.save_gradients",
    "tl.OpLog.save_tensor_data",
    "tl.OpLog.scalar_bool_value",
    "tl.OpLog.show",
    "tl.OpLog.sibling_layers",
    "tl.OpLog.source_trace",
    "tl.OpLog.tensor",
    "tl.OpLog.tensor_dtype",
    "tl.OpLog.tensor_label_raw",
    "tl.OpLog.tensor_memory",
    "tl.OpLog.tensor_memory_str",
    "tl.OpLog.tensor_shape",
    "tl.OpLog.transformed_activation",
    "tl.OpLog.transformed_activation_dtype",
    "tl.OpLog.transformed_activation_memory",
    "tl.OpLog.transformed_activation_shape",
    "tl.OpLog.transformed_gradient",
    "tl.OpLog.transformed_gradient_dtype",
    "tl.OpLog.transformed_gradient_memory",
    "tl.OpLog.transformed_gradient_shape",
    "tl.OpLog.uses_params",
]

audit_touch_items("Remaining pass fields", ITEMS, AUDIT_CONTEXT)

In [ ]:
COVERAGE_CALLS = [
    "tl.OpLog",
    "tl.OpLog.DEFAULT_FILL_STATE",
    "tl.OpLog.FORK_POLICY",
    "tl.OpLog.PORTABLE_STATE_SPEC",
    "tl.OpLog._construction_done",
    "tl.OpLog._pass_finished",
    "tl.OpLog.activation",
    "tl.OpLog.activation_postfunc",
    "tl.OpLog.activation_transform",
    "tl.OpLog.args_captured",
    "tl.OpLog.autograd_saved_bytes",
    "tl.OpLog.autograd_saved_tensor_count",
    "tl.OpLog.bool_conditional_id",
    "tl.OpLog.bool_context_kind",
    "tl.OpLog.bool_is_branch",
    "tl.OpLog.bool_wrapper_kind",
    "tl.OpLog.buffer_address",
    "tl.OpLog.buffer_parent",
    "tl.OpLog.buffer_pass",
    "tl.OpLog.bytes_delta_at_call",
    "tl.OpLog.bytes_peak_at_call",
    "tl.OpLog.captured_arg_template",
    "tl.OpLog.captured_args",
    "tl.OpLog.captured_kwarg_template",
    "tl.OpLog.captured_kwargs",
    "tl.OpLog.child_layers",
    "tl.OpLog.children_tensor_versions",
    "tl.OpLog.co_parent_layers",
    "tl.OpLog.cond_branch_children_by_cond",
    "tl.OpLog.cond_branch_elif_children",
    "tl.OpLog.cond_branch_else_children",
    "tl.OpLog.cond_branch_start_children",
    "tl.OpLog.cond_branch_then_children",
    "tl.OpLog.conditional_branch_depth",
    "tl.OpLog.conditional_branch_stack",
    "tl.OpLog.container_spec",
    "tl.OpLog.containing_module",
    "tl.OpLog.containing_modules",
    "tl.OpLog.copy",
    "tl.OpLog.corresponding_grad_fn",
    "tl.OpLog.creation_order",
    "tl.OpLog.detach_saved_tensor",
    "tl.OpLog.edge_uses",
    "tl.OpLog.equivalent_operations",
    "tl.OpLog.extra_data",
    "tl.OpLog.feeds_output",
    "tl.OpLog.flops_backward",
    "tl.OpLog.flops_forward",
    "tl.OpLog.func_applied",
    "tl.OpLog.func_argnames",
    "tl.OpLog.func_autocast_state",
    "tl.OpLog.func_call_id",
    "tl.OpLog.func_call_stack",
    "tl.OpLog.func_config",
    "tl.OpLog.func_is_inplace",
    "tl.OpLog.func_kwargs_non_tensor",
    "tl.OpLog.func_name",
    "tl.OpLog.func_non_tensor_args",
    "tl.OpLog.func_positional_args_non_tensor",
    "tl.OpLog.func_rng_states",
    "tl.OpLog.func_time",
    "tl.OpLog.get_child_layers",
    "tl.OpLog.get_parent_layers",
    "tl.OpLog.grad_dtype",
    "tl.OpLog.grad_fn_id",
    "tl.OpLog.grad_fn_name",
    "tl.OpLog.grad_fn_object",
    "tl.OpLog.grad_memory",
    "tl.OpLog.grad_memory_str",
    "tl.OpLog.grad_shape",
    "tl.OpLog.gradient",
    "tl.OpLog.has_child_tensor_variations",
    "tl.OpLog.has_children",
    "tl.OpLog.has_co_parents",
    "tl.OpLog.has_gradient",
    "tl.OpLog.has_input_ancestor",
    "tl.OpLog.has_internally_initialized_ancestor",
    "tl.OpLog.has_parents",
    "tl.OpLog.has_saved_activations",
    "tl.OpLog.has_siblings",
    "tl.OpLog.in_cond_branch",
    "tl.OpLog.input_ancestors",
    "tl.OpLog.internally_initialized_ancestors",
    "tl.OpLog.internally_initialized_parents",
    "tl.OpLog.intervention_log",
    "tl.OpLog.io_role",
    "tl.OpLog.is_buffer_layer",
    "tl.OpLog.is_computed_inside_submodule",
    "tl.OpLog.is_final_output",
    "tl.OpLog.is_input_layer",
    "tl.OpLog.is_internally_initialized",
    "tl.OpLog.is_internally_terminated",
    "tl.OpLog.is_leaf_module_output",
    "tl.OpLog.is_output_ancestor",
    "tl.OpLog.is_output_layer",
    "tl.OpLog.is_part_of_iterable_output",
    "tl.OpLog.is_scalar_bool",
    "tl.OpLog.is_submodule_input",
    "tl.OpLog.is_submodule_output",
    "tl.OpLog.is_terminal_bool_layer",
    "tl.OpLog.iterable_output_index",
    "tl.OpLog.layer_label",
    "tl.OpLog.layer_label_no_pass",
    "tl.OpLog.layer_label_no_pass_short",
    "tl.OpLog.layer_label_raw",
    "tl.OpLog.layer_label_short",
    "tl.OpLog.layer_label_w_pass",
    "tl.OpLog.layer_label_w_pass_short",
    "tl.OpLog.layer_total_num",
    "tl.OpLog.layer_type",
    "tl.OpLog.layer_type_num",
    "tl.OpLog.leaf_module_pass",
    "tl.OpLog.log_tensor_grad",
    "tl.OpLog.lookup_keys",
    "tl.OpLog.macs_backward",
    "tl.OpLog.macs_forward",
    "tl.OpLog.materialize_activation",
    "tl.OpLog.materialize_gradient",
    "tl.OpLog.max_distance_from_input",
    "tl.OpLog.max_distance_from_output",
    "tl.OpLog.min_distance_from_input",
    "tl.OpLog.min_distance_from_output",
    "tl.OpLog.module_address_normalized",
    "tl.OpLog.module_entry_exit_thread_output",
    "tl.OpLog.module_entry_exit_threads_inputs",
    "tl.OpLog.module_nesting_depth",
    "tl.OpLog.module_passes_entered",
    "tl.OpLog.module_passes_exited",
    "tl.OpLog.modules_entered",
    "tl.OpLog.modules_entered_argnames",
    "tl.OpLog.modules_exited",
    "tl.OpLog.num_args",
    "tl.OpLog.num_keyword_args",
    "tl.OpLog.num_param_tensors",
    "tl.OpLog.num_params_frozen",
    "tl.OpLog.num_params_total",
    "tl.OpLog.num_params_trainable",
    "tl.OpLog.num_passes",
    "tl.OpLog.num_positional_args",
    "tl.OpLog.operation_equivalence_type",
    "tl.OpLog.operation_num",
    "tl.OpLog.output_descendants",
    "tl.OpLog.output_device",
    "tl.OpLog.output_path",
    "tl.OpLog.params",
    "tl.OpLog.params_memory",
    "tl.OpLog.params_memory_str",
    "tl.OpLog.parent_layer_arg_locs",
    "tl.OpLog.parent_layers",
    "tl.OpLog.parent_param_barcodes",
    "tl.OpLog.parent_param_logs",
    "tl.OpLog.parent_param_passes",
    "tl.OpLog.parent_param_shapes",
    "tl.OpLog.parent_params",
    "tl.OpLog.pass_num",
    "tl.OpLog.passes",
    "tl.OpLog.recurrent_group",
    "tl.OpLog.root_ancestors",
    "tl.OpLog.save_gradients",
    "tl.OpLog.save_tensor_data",
    "tl.OpLog.scalar_bool_value",
    "tl.OpLog.show",
    "tl.OpLog.sibling_layers",
    "tl.OpLog.source_trace",
    "tl.OpLog.tensor",
    "tl.OpLog.tensor_dtype",
    "tl.OpLog.tensor_label_raw",
    "tl.OpLog.tensor_memory",
    "tl.OpLog.tensor_memory_str",
    "tl.OpLog.tensor_shape",
    "tl.OpLog.transformed_activation",
    "tl.OpLog.transformed_activation_dtype",
    "tl.OpLog.transformed_activation_memory",
    "tl.OpLog.transformed_activation_shape",
    "tl.OpLog.transformed_gradient",
    "tl.OpLog.transformed_gradient_dtype",
    "tl.OpLog.transformed_gradient_memory",
    "tl.OpLog.transformed_gradient_shape",
    "tl.OpLog.uses_params",
]
print(f"coverage markers: {len(COVERAGE_CALLS)}")

In [ ]:
if TMPDIR.exists():
    shutil.rmtree(TMPDIR)
print("cleanup complete")